In [1]:
import pandas as pd
import requests
import datetime
import time

def get_binance_klines(symbol="BTCUSDT", interval="1h", start="2017-01-01"):# skida istorijske Bitcoin cene sa Binance API.
    """
    Download historical OHLCV data from Binance.
    interval examples: 1m, 5m, 15m, 1h, 4h, 1d
    """
    
    url = "https://api.binance.com/api/v3/klines"# endpoint za price data.
    start_ts = int(pd.Timestamp(start).timestamp() * 1000)# pretvara datum u timestamp.
    end_ts = int(time.time() * 1000)

    all_data = []

    while start_ts < end_ts:# Skida podatke u batch-evima jer API ima limit.
        params = {
            "symbol": symbol,
            "interval": interval,
            "startTime": start_ts,
            "limit": 1000  # Binance max per request
        }

        data = requests.get(url, params=params).json()

        # Stop if Binance returns empty list (end of available data)
        if not data:
            break

        all_data.extend(data)

        # Move start to last returned timestamp + 1ms
        last_time = data[-1][0]
        start_ts = last_time + 1

        # avoid hitting rate limit
        time.sleep(0.4)

    # Convert into DataFrame
    df = pd.DataFrame(all_data, columns=[
        "OpenTime", "Open", "High", "Low", "Close", "Volume",
        "CloseTime", "QuoteVolume", "Trades", "TakerBuyBase",
        "TakerBuyQuote", "Ignore"
    ])

    # Clean up
    df["OpenTime"] = pd.to_datetime(df["OpenTime"], unit="ms")
    df = df.set_index("OpenTime")

    df = df[["Open", "High", "Low", "Close", "Volume"]].astype(float)

    # Rename to match yfinance naming style
    df.index.name = "Datetime"

    return df

In [2]:
df = get_binance_klines("BTCUSDT", interval="1h", start="2017-01-01")# skida Bitcoin cenu po satu od 2017.
df.head()# prikaz prvih redova

,Open,High,Low,Close,Volume
Datetime,,,,,
2017-08-17 04:00:00,4261.48,4313.62,4261.32,4308.83,47.181009
2017-08-17 05:00:00,4308.83,4328.69,4291.37,4315.32,23.234916
2017-08-17 06:00:00,4330.29,4345.45,4309.37,4324.35,7.229691
2017-08-17 07:00:00,4316.62,4349.99,4287.41,4349.99,4.443249
2017-08-17 08:00:00,4333.32,4377.85,4333.32,4360.69,0.972807


In [3]:
#!pip install ta
import ta

#return
df["return_1h"] = df["Close"].pct_change()# koliko se cena promenila u % zato što model lakše uči promene nego apsolutne cene.
df['daily_return'] = df['Close'].pct_change(24)# Dnevni prinos
#mean/std
df["rolling_mean_24h"] = df["Close"].rolling(24).mean()# prosečna cena zadnja 24h
df["rolling_std_24h"] = df["Close"].rolling(24).std()# volatilnost
#lag
df["close_lag_6h"] = df["Close"].shift(6)# cena pre 6h
df["close_lag_12h"] = df["Close"].shift(12)# cena pre 12h
df["close_lag_24h"] = df["Close"].shift(24)# cena pre 24h
df['close_lag_48h'] = df['Close'].shift(48)  # Cena pre 48h
df['close_lag_168h'] = df['Close'].shift(168)  # Cena pre 7 dana

# RSI
df['RSI_14'] = ta.momentum.RSIIndicator(df['Close'], window=14).rsi()

# MACD
macd = ta.trend.MACD(df['Close'], window_slow=26, window_fast=12, window_sign=9)
df['MACD'] = macd.macd()
df['MACD_signal'] = macd.macd_signal()  # MACD signal linija
df['MACD_hist'] = macd.macd_diff()  # MACD histogram

# Bollinger Bands
bb = ta.volatility.BollingerBands(df['Close'], window=20, window_dev=2)
df['BB_mavg'] = bb.bollinger_mavg()  # Srednja Bollinger linija
df['BB_upper'] = bb.bollinger_hband()  # Gornja Bollinger linija
df['BB_lower'] = bb.bollinger_lband()  # Donja Bollinger linija

# SMA i EMA
df['SMA_20'] = ta.trend.SMAIndicator(df['Close'], window=20).sma_indicator()
df['EMA_20'] = ta.trend.EMAIndicator(df['Close'], window=20).ema_indicator()

# ATR (Average True Range)
df['ATR_14'] = ta.volatility.AverageTrueRange(high=df['High'], low=df['Low'], close=df['Close'], window=14).average_true_range()

# ROC (Rate of Change) - Procenat promene cene u poslednjem periodu
df['ROC'] = ta.momentum.ROCIndicator(df['Close'], window=12).roc()

df["volatility_24h"] = df["return_1h"].rolling(24).std()

# za regresiju
df["price_24h"] = df["Close"].shift(-24)
df["price_48h"] = df["Close"].shift(-48)
df["price_7d"] = df["Close"].shift(-168)

df = df.dropna()# brisanje jer rolling i lag nekada stvaraju prazne vrednosti.
df.head() #ispis :)

,Open,High,Low,Close,Volume,return_1h,daily_return,rolling_mean_24h,rolling_std_24h,close_lag_6h,...,BB_upper,BB_lower,SMA_20,EMA_20,ATR_14,ROC,volatility_24h,price_24h,price_48h,price_7d
Datetime,,,,,,,,,,,,,,,,,,,,,
2017-08-24 04:00:00,4113.58,4148.19,4090.39,4113.98,32.247571,0.000097,0.007440,4145.985833,52.439849,4114.20,...,4249.760962,4067.399038,4158.5800,4123.466923,73.916662,-2.281456,0.006937,4310.20,4297.94,4591.56
2017-08-24 05:00:00,4113.98,4177.64,4113.49,4132.09,28.158769,0.004402,0.019469,4149.273750,48.709115,4114.01,...,4244.510883,4065.546117,4155.0285,4124.288168,73.219043,-0.820398,0.006779,4281.04,4295.75,4598.54
2017-08-24 06:00:00,4132.09,4177.18,4131.91,4133.42,29.921536,0.000322,0.013093,4151.499583,46.579934,4131.00,...,4233.637800,4066.959200,4150.2985,4125.157866,71.222683,-0.240143,0.006666,4302.72,4319.70,4572.99
2017-08-24 07:00:00,4153.97,4173.99,4133.41,4153.32,32.851584,0.004814,0.019250,4154.767917,43.628501,4140.91,...,4219.123982,4073.006018,4146.0650,4127.839974,69.033920,0.880481,0.006709,4329.00,4319.70,4592.16
2017-08-24 08:00:00,4153.80,4206.88,4153.32,4200.00,32.275428,0.011239,0.018429,4157.934583,44.054251,4131.92,...,4215.620914,4074.941086,4145.2810,4134.712358,67.928640,1.981352,0.006652,4332.17,4319.70,4586.51


In [4]:
# -----------------------------
# 1) TEMPORAL SPLIT
# -----------------------------


# izdvajamo target
target_col = "price_24h"# za sad 24h, 48h i 7d kasnije treba za test najboljeg mozela ili bar kada uspe 24h da dobije dobar rezultat prvo


features = [
    "Open","High","Low","Close","Volume",

    "return_1h",
    "daily_return",

    "rolling_mean_24h",
    "rolling_std_24h",

    "close_lag_6h",
    "close_lag_12h",
    "close_lag_24h",
    "close_lag_48h",
    "close_lag_168h",

    "RSI_14",
    "MACD",
    "MACD_signal",
    "MACD_hist",

    "ATR_14",
    "ROC",

    "SMA_20",
    "EMA_20",
    "BB_upper",
    "BB_lower"
]

# indeks za split (60/20/20)
n = len(df)
train_end = int(n * 0.6)
val_end = int(n * 0.8)

#najstariji podaci idu za train, onda za vel i na kraju test
train_df = df.iloc[:train_end]
val_df   = df.iloc[train_end:val_end]
test_df  = df.iloc[val_end:]

print(len(train_df), len(val_df), len(test_df))

44644 14881 14882


In [5]:
# -----------------------------
# 2) SCALING
# -----------------------------

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# FIT samo na train
train_df[features] = scaler.fit_transform(train_df[features])

# TRANSFORM na val i test
val_df[features] = scaler.transform(val_df[features])
test_df[features] = scaler.transform(test_df[features])

In [14]:
# Importovanje potrebnih biblioteka
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# -----------------------------
# 1) Priprema podataka
# -----------------------------

# X i y za train/val/test (isti kao kod RF)
X_train = train_df[features].values
y_train = train_df[target_col].values

X_val = val_df[features].values
y_val = val_df[target_col].values

X_test = test_df[features].values
y_test = test_df[target_col].values

# -----------------------------
# 2) Linear Regression
# -----------------------------

lr = LinearRegression()

# Treniranje modela
lr.fit(X_train, y_train)

# Predikcija na test setu
y_pred = lr.predict(X_test)

# -----------------------------
# Linear Regression evaluacija (regresija)
# -----------------------------

# Predikcija na validation setu
val_preds = lr.predict(X_val)
val_true = y_val

# RMSE
rmse = np.sqrt(mean_squared_error(val_true, val_preds))
# MAE
mae = mean_absolute_error(val_true, val_preds)
# R²
r2 = r2_score(val_true, val_preds)
# MAPE
mape = np.mean(np.abs((val_true - val_preds) / val_true)) * 100

print("\n-------------")
print("Linear Regression Evaluation on Validation Set")
print(f"MAE: {mae:.4f} | RMSE: {rmse:.4f} | MAPE: {mape:.2f}% | R²: {r2:.4f}")
print("Mean of predictions:", np.mean(val_preds))
print("Mean of true values:", np.mean(val_true))
print("Target mean (train):", np.mean(y_train))
print("Target std (train):", np.std(y_train))


-------------
Linear Regression Evaluation on Validation Set
MAE: 626.6918 | RMSE: 1017.2092 | MAPE: 1.69% | R²: 0.9962
Mean of predictions: 35210.11031583198
Mean of true values: 35366.542036825485
Target mean (train): 19568.47096989517
Target std (train): 17131.72400263255
